# Territory KPIs (weekly)

Quick analysis notebook - started in pandas, moved some bits to Spark when it got slow. Run top to bottom. TODO: clean up before handing to data eng.

In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = SparkSession.builder.appName('territory_kpis_adhoc').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
DATA = '../data'  # TODO parametrise

In [ ]:
# load everything - some in pandas, some in spark, whatever was quicker at the time
visits = pd.read_parquet(DATA + '/visits.parquet')
fu = pd.read_parquet(DATA + '/follow_ups.parquet')
sales_sdf = spark.read.parquet(DATA + '/weekly_sales.parquet')
reps = pd.read_parquet(DATA + '/reps.parquet')
terr = pd.read_parquet(DATA + '/territories.parquet')
print(len(visits), len(fu), sales_sdf.count())

In [ ]:
# week = monday of the visit week
visits['visit_date'] = pd.to_datetime(visits['visit_date'])
visits['week'] = (visits['visit_date'] - pd.to_timedelta(visits['visit_date'].dt.weekday, unit='D')).dt.date
visits['f2f'] = (visits['channel'] == 'face_to_face').astype(int)
v = visits.groupby(['territory_id', 'week']).agg(
    visits=('visit_id', 'count'),
    unique_hcps=('hcp_id', 'nunique'),
    f2f_visits=('f2f', 'sum'),
    avg_duration_min=('duration_min', 'mean'),
).reset_index()
v['f2f_share'] = (v['f2f_visits'] / v['visits']).round(4)
v['avg_duration_min'] = v['avg_duration_min'].round(2)
v.head()

In [ ]:
# open follow ups due in week -> need territory, join via reps (follow_ups has no territory col!)
fu['due_date'] = pd.to_datetime(fu['due_date'])
fu['week'] = (fu['due_date'] - pd.to_timedelta(fu['due_date'].dt.weekday, unit='D')).dt.date
fu2 = fu.merge(reps[['rep_id', 'territory_id']], on='rep_id', how='left')
open_fu = fu2[fu2['status'] == 'open'].groupby(['territory_id', 'week']).size().reset_index(name='open_follow_ups')

In [ ]:
# sales in spark because the file was big in prod
sales = (sales_sdf.groupBy('territory_id', 'week')
         .agg(F.sum('units').alias('units'), F.round(F.sum('revenue_gbp'), 2).alias('revenue_gbp'))
         .toPandas())
sales['week'] = pd.to_datetime(sales['week']).dt.date

In [ ]:
# reps per territory for visits_per_rep
rep_cnt = reps.groupby('territory_id').size().reset_index(name='rep_count')

In [ ]:
kpis = sales.merge(v, on=['territory_id', 'week'], how='left') \
            .merge(open_fu, on=['territory_id', 'week'], how='left') \
            .merge(rep_cnt, on='territory_id', how='left') \
            .merge(terr[['territory_id', 'region']], on='territory_id', how='left')
for c in ['visits', 'unique_hcps', 'f2f_visits', 'open_follow_ups']:
    kpis[c] = kpis[c].fillna(0).astype(int)
kpis['f2f_share'] = kpis['f2f_share'].fillna(0.0)
kpis['avg_duration_min'] = kpis['avg_duration_min'].fillna(0.0)
kpis['visits_per_rep'] = (kpis['visits'] / kpis['rep_count']).round(2)
kpis['revenue_per_visit'] = np.where(kpis['visits'] > 0, (kpis['revenue_gbp'] / kpis['visits']).round(2), 0.0)
kpis = kpis[['territory_id', 'region', 'week', 'visits', 'unique_hcps', 'f2f_visits', 'f2f_share', 'avg_duration_min',
             'open_follow_ups', 'rep_count', 'visits_per_rep', 'units', 'revenue_gbp', 'revenue_per_visit']]
kpis = kpis.sort_values(['territory_id', 'week']).reset_index(drop=True)
print(kpis.shape)
kpis.head(10)

In [ ]:
# save for the dashboard team
kpis['week'] = pd.to_datetime(kpis['week'])
kpis.to_parquet('expected_output/territory_kpis.parquet', index=False)
kpis.to_csv('expected_output/territory_kpis.csv', index=False)